# 🐱🐶 Cats vs Dogs — CNN from Scratch
**| Custom CNN | TensorFlow / Keras**


## 1. Imports

In [ ]:
import kagglehub
kagglehub.login()

In [ ]:
import os, shutil, pathlib

KAGGLE_DATASET = "aleemaparakatta/cats-and-dogs-mini-dataset"
LOCAL_DATA_DIR = "data"

try:
    import kagglehub
    dl_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_DATASET))
    def find_root(base):
        for root, dirs, _ in os.walk(base):
            if "cat_set" in dirs or "dog_set" in dirs:
                return pathlib.Path(root)
        return base
    data_root = find_root(dl_path)
    if not os.path.exists(LOCAL_DATA_DIR):
        shutil.copytree(str(data_root), LOCAL_DATA_DIR)
    print(f"✓ Data ready at ./{LOCAL_DATA_DIR}")
except ImportError:
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    os.system(f"kaggle datasets download -d {KAGGLE_DATASET} --unzip -p {LOCAL_DATA_DIR}")

for cls in sorted(pathlib.Path(LOCAL_DATA_DIR).iterdir()):
    if cls.is_dir():
        print(f"  {cls.name}/  — {len(list(cls.glob('*.*')))} images")

In [ ]:
import os, json, random, pathlib, shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
print(f"TensorFlow : {tf.__version__}")

## 2. Configuration

In [ ]:
DATA_DIR   = "data"
SPLIT_DIR  = "data_split"
OUTPUT_DIR = "outputs_scratch_v2"
IMG_SIZE   = (128, 128)
BATCH_SIZE = 16     # smaller batch → more gradient updates per epoch
EPOCHS     = 60     # more epochs → more time to learn
LR         = 3e-4   # lower LR → more stable training
L2         = 1e-4   # weight decay
SEED       = 42
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Prepare Train / Val / Test Split


In [ ]:
EXTS = {".jpg",".jpeg",".png",".bmp",".webp"}

def build_split(src, dst, val_f=0.15, test_f=0.15, seed=42):
    if pathlib.Path(dst).exists():
        print(f"'{dst}' exists — skipping."); return
    random.seed(seed)
    for cls in sorted(pathlib.Path(src).iterdir()):
        if not cls.is_dir(): continue
        files = [f for f in cls.iterdir() if f.suffix.lower() in EXTS]
        random.shuffle(files)
        n = len(files); nt=int(n*test_f); nv=int(n*val_f)
        for split, fl in [("test",files[:nt]),("val",files[nt:nt+nv]),("train",files[nt+nv:])]:
            d = pathlib.Path(dst)/split/cls.name; d.mkdir(parents=True, exist_ok=True)
            for f in fl: shutil.copy(f, d/f.name)
        print(f"  {cls.name}: train={len(files[nt+nv:])} val={nv} test={nt}")

build_split(DATA_DIR, SPLIT_DIR)

## 4. Build tf.data Pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_ds(folder, shuffle=False):
    return tf.keras.utils.image_dataset_from_directory(
        folder, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        shuffle=shuffle, seed=SEED, label_mode="binary")

train_ds = load_ds(os.path.join(SPLIT_DIR,"train"), shuffle=True)
val_ds   = load_ds(os.path.join(SPLIT_DIR,"val"))
test_ds  = load_ds(os.path.join(SPLIT_DIR,"test"))
CLASS_NAMES = train_ds.class_names
print(f"Classes: {CLASS_NAMES}")

# Heavy augmentation — key to fighting small dataset
augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.25),
    layers.RandomContrast(0.3),
    layers.RandomBrightness(0.2),
    layers.RandomTranslation(0.1, 0.1),
], name="heavy_augmentation")

def pre_train(x, y): return augment(tf.cast(x,tf.float32)/255., training=True), y
def pre_eval(x, y):  return tf.cast(x,tf.float32)/255., y

train_ds = train_ds.map(pre_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(pre_eval,   num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds  = test_ds.map(pre_eval,  num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
print("✓ Pipeline ready.")

## 5. Build Custom CNN from Scratch
Classic Conv → BN → Pool blocks, no pretrained weights.

In [ ]:
def build_improved_cnn(input_shape=(*IMG_SIZE,3), l2=L2):
    reg = regularizers.l2(l2)
    return keras.Sequential([
        keras.Input(shape=input_shape),

        # Block 1 — 32 filters
        layers.Conv2D(32,3,padding="same",kernel_regularizer=reg),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.Conv2D(32,3,padding="same",kernel_regularizer=reg),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2), layers.Dropout(0.3),

        # Block 2 — 64 filters
        layers.Conv2D(64,3,padding="same",kernel_regularizer=reg),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.Conv2D(64,3,padding="same",kernel_regularizer=reg),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2), layers.Dropout(0.3),

        # Block 3 — 128 filters
        layers.Conv2D(128,3,padding="same",kernel_regularizer=reg),
        layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2), layers.Dropout(0.4),

        # Head
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu", kernel_regularizer=reg),
        layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid"),
    ], name="improved_cnn")

model = build_improved_cnn()
model.summary()
total = sum(np.prod(v.shape) for v in model.trainable_weights)
print(f"\nTotal params: {total:,.0f}")

## 6. Compile & Train

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=0.1),  # label smoothing
    metrics=["accuracy"],
)

cbs = [
    keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR,"best_model.keras"),
        monitor="val_accuracy", save_best_only=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=6, min_lr=1e-7, verbose=1),
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=15, restore_best_weights=True, verbose=1),
]

history = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS, callbacks=cbs
)
hist_dict = {k:[float(v) for v in vals] for k,vals in history.history.items()}
with open(os.path.join(OUTPUT_DIR,"history.json"),"w") as f:
    json.dump(hist_dict, f, indent=2)
print("✓ Training complete.")

## 7. Plot Training History

In [ ]:
h  = history.history
ep = range(1, len(h["loss"])+1)
fig, axes = plt.subplots(1,2,figsize=(13,4))
fig.suptitle("Improved Scratch CNN — Training History", fontsize=14, fontweight="bold")
axes[0].plot(ep,h["loss"],"b-",label="Train"); axes[0].plot(ep,h["val_loss"],"r-",label="Val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep,h["accuracy"],"b-",label="Train"); axes[1].plot(ep,h["val_accuracy"],"r-",label="Val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].set_ylim(0,1)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_:f"{y:.0%}"))
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"history.png"),dpi=150)
plt.show()

## 8. Evaluate on TEST Set

In [ ]:
best = keras.models.load_model(os.path.join(OUTPUT_DIR,"best_model.keras"))
loss, acc = best.evaluate(test_ds, verbose=1)
print(f"\nTest Loss     : {loss:.4f}")
print(f"Test Accuracy : {acc:.2%}")

y_true, y_pred = [], []
for imgs, labels in test_ds:
    probs = best.predict(imgs, verbose=0)
    preds = (probs.squeeze()>0.5).astype(int)
    y_true.extend(labels.numpy().astype(int).flatten())
    y_pred.extend(preds.flatten())
y_true, y_pred = np.array(y_true), np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix — Test Set", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"confusion_matrix.png"),dpi=150)
plt.show()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))